In [ ]:
# !pip install ninja --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
#  !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# !pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# !pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.13.0+cu126
CUDA Available: True
CUDA Version: 12.6
GPU Name: NVIDIA L40S
VRAM: 44.4 GB


In [4]:
MODEL_ID = "empero-ai/Qwen3.8-2B-Distill"

OUTPUT_BASE_DIR = "./Qwen3.8-2B-Distill"
LOCAL_PATH = "./local_model_Qwen3.8-2B-Distill"

In [5]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 130.03it/s]
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model_Qwen3.8-2B-Distill
Download complete: :                                       |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [6]:
import os

from safetensors import safe_open

for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors
mtp.fc.weight
mtp.layers.0.input_layernorm.weight
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.q_proj.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

In [8]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [9]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [10]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 1000,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 4,  # Faster on 48GB VRAM
    "seqlen": 2048*2,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": {
            "mtp": {"data_type": "bfloat16"},
            "mtp.fc": {"data_type": "bfloat16"}
        }
}

In [11]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-08-31 09:46:56 WARNING autoround.py L592: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-31 09:46:56 WARNING autoround.py L592: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-31 09:46:56 WARNING autoround.py L592: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [12]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,llm_compressor", inplace=True
)

2026-08-31 09:47:03 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22].linear_attn.in_proj_a, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22].linear_attn.in_proj_b
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-08-31 09:47:03 INFO base.py L1303: `torch.compile` is enabled
2026-08-31 09:47:04 INFO orchestrator.py L570: start to cache block inputs
2026-08-31 09:47:04 INFO mllm.py L86: Using MLLM template: qwen3_5
2026-08-31 09:47:04 INFO calib_dataset.py L1113: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-08-31 09:47:21 INFO device.py L1448: 'peak_ram': 14.01GB, 'peak_vram': 4.23GB
2026-08-31 09:47:21 INFO orchestrator.py L602: caching done
Quantizing model.language_model.layers.0:   0%|          | 0/24 [00:03<?, ?it/s]2026-08-31 09:48:27 INFO quantizer.py L545: U

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-31 10:11:48 INFO missing_tensors.py L373: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 1/1 [00:00<00:00, 38.00shard/s]
2026-08-31 10:11:48 INFO missing_tensors.py L861: Processing config.json to update quantization_config for missing tensors...
2026-08-31 10:11:48 INFO missing_tensors.py L828: Updated extra_config for 8 ignored layer(s): mtp.fc, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.self_attn.k_proj, m

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-31 10:12:04 INFO missing_tensors.py L373: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 1/1 [00:00<00:00, 107.42shard/s]
2026-08-31 10:12:04 INFO missing_tensors.py L513: Successfully wrote 15 missing tensor(s) to 'model_extra_tensors.safetensors' in ./Qwen3.8-2B-Distill/local_model_Qwen3.8-2B-Distill-w4g32/auto-gptq.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-31 10:12:50 INFO missing_tensors.py L373: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 1/1 [00:00<00:00, 159.98shard/s]
2026-08-31 10:12:50 INFO missing_tensors.py L513: Successfully wrote 15 missing tensor(s) to 'model_extra_tensors.safetensors' in ./Qwen3.8-2B-Distill/local_model_Qwen3.8-2B-Distill-w4g32/llm-compressor-wint-a16.
2026-08-31 10:12:51 INFO device.py L1448: 'peak_ram': 18.22GB, 'peak_vram': 26.35GB


(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1024)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-23): 24 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1024, out_features=3072, bias=True)
             (proj): Linear(in_features=1024, out_features=1024, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
             (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
             (act_fn): 

In [13]:
notebook_login()

In [14]:
HF_USER = "Vishva007"

In [15]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [16]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model_Qwen3.8-2B-Distill-w4g32/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model_Qwen3.8-2B-Distill-w4g32/auto-gptq"), 
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
    push_to_hub(
            os.path.join(OUTPUT_BASE_DIR, "local_model_Qwen3.8-2B-Distill-w4g32/llm-compressor-wint-a16"), 
            f"{base_name}-W4A16-AutoRound-LLM-Compressor", 
            hf_token)
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./Qwen3.8-2B-Distill/local_model_Qwen3.8-2B-Distill-w4g32/auto-round-auto-gptq to Vishva007/Qwen3.8-2B-Distill-W4A16-AutoRound...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.8-2B-Distill-W4A16-AutoRound

[Hub] Pushing ./Qwen3.8-2B-Distill/local_model_Qwen3.8-2B-Distill-w4g32/auto-gptq to Vishva007/Qwen3.8-2B-Distill-W4A16-AutoRound-GPTQ...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.8-2B-Distill-W4A16-AutoRound-GPTQ

[Hub] Pushing ./Qwen3.8-2B-Distill/local_model_Qwen3.8-2B-Distill-w4g32/llm-compressor-wint-a16 to Vishva007/Qwen3.8-2B-Distill-W4A16-AutoRound-LLM-Compressor...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.8-2B-Distill-W4A16-AutoRound-LLM-Compressor
